In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [2]:
i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.

In [3]:
dir = "E:\Downloads\Telegram Desktop\plant_disease\healthy"

In [4]:
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    

Found 8688 files belonging to 8 classes.
Using 6951 files for training.


In [5]:
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set

Found 8688 files belonging to 8 classes.
Using 1737 files for validation.


In [6]:
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)


In [7]:
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())

Classes: ['Apple___healthy', 'Corn___Healthy', 'Pepper__bell___healthy', 'Potato___Healthy', 'Rice___Healthy', 'Sugarcane_Healthy', 'Tomato_healthy', 'Wheat___Healthy']
Train batches: 218
Val batches: 28
Test batches: 27


In [8]:
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)

In [9]:
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [10]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers


In [11]:
num = len(class_names)


this is the base of the model that uses efficientnetb for transfer learning


In [12]:
base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False

In [14]:
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint

In [15]:
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm


In [16]:
x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)

In [17]:
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)

In [18]:
model = models.Model(inputs, outputs)

In [19]:
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]

In [21]:
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8023 - loss: 0.6535

218/218 ━━━━━━━━━━━━━━━━━━━━ 376s 1s/step - accuracy: 0.9219 - loss: 0.2508 - val_accuracy: 0.9954 - val_loss: 0.0516 - learning_rate: 0.0010
Epoch 2/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9876 - loss: 0.0480

218/218 ━━━━━━━━━━━━━━━━━━━━ 397s 1s/step - accuracy: 0.9888 - loss: 0.0438 - val_accuracy: 0.9954 - val_loss: 0.0150 - learning_rate: 0.0010
Epoch 3/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9949 - loss: 0.0237

218/218 ━━━━━━━━━━━━━━━━━━━━ 332s 1s/step - accuracy: 0.9927 - loss: 0.0275 - val_accuracy: 0.9977 - val_loss: 0.0075 - learning_rate: 0.0010
Epoch 4/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9908 - loss: 0.0270

218/218 ━━━━━━━━━━━━━━━━━━━━ 345s 1s/step - accuracy: 0.9918 - loss: 0.0259 - val_accuracy: 0.9977 - val_loss: 0.0041 - learning_rate: 0.0010
Epoch 5/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.9940 - loss: 0.0204 - val_accuracy: 0.9977 - val_loss: 0.0050 - learning_rate: 0.0010
Epoch 6/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9926 - loss: 0.0196

218/218 ━━━━━━━━━━━━━━━━━━━━ 351s 1s/step - accuracy: 0.9937 - loss: 0.0181 - val_accuracy: 0.9989 - val_loss: 0.0025 - learning_rate: 0.0010
Epoch 7/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 327s 1s/step - accuracy: 0.9944 - loss: 0.0161 - val_accuracy: 0.9966 - val_loss: 0.0058 - learning_rate: 0.0010
Epoch 8/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 326s 1s/step - accuracy: 0.9953 - loss: 0.0150 - val_accuracy: 0.9989 - val_loss: 0.0025 - learning_rate: 0.0010
Epoch 9/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 330s 1s/step - accuracy: 0.9958 - loss: 0.0153 - val_accuracy: 0.9989 - val_loss: 0.0053 - learning_rate: 3.0000e-04


In [22]:
model.save("healthy.keras")

In [31]:
import numpy as np
def predict_plant(img_path):
    img = keras.utils.load_img(img_path, target_size=(224, 224))
    img_array = keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])
    
    print(f"Result: {class_names[np.argmax(score)]}")

predict_plant(r"C:\Users\dedha\Downloads\sugar.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 452ms/step
Result: Sugarcane_Healthy


In [32]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

27/27 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.9988 - loss: 0.0026
Test Loss: 0.0026
Test Accuracy: 99.88%


This is not good to use